In [41]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("crowww/a-large-scale-fish-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'a-large-scale-fish-dataset' dataset.
Path to dataset files: /kaggle/input/a-large-scale-fish-dataset


In [42]:
from pathlib import Path
import tensorflow as tf
import shutil

In [43]:
data_dir = Path('/content/a-large-scale-fish-dataset')

# ตรวจสอบและลบไฟล์ขยะหรือไฟล์รูปที่พัง เพื่อจัดการความไม่สมบูรณ์ของข้อมูล
num_skipped = 0
removed_count = 0
removed_folders = 0

for file_path in data_dir.rglob('*'):
    if file_path.is_file():
        if file_path.suffix.lower() not in ['.png', '.jpg', '.jpeg']:
            file_path.unlink()
            removed_count += 1

for folder_path in data_dir.rglob('*GT*'):
    if folder_path.is_dir() and folder_path.exists():
        shutil.rmtree(folder_path)
        removed_folders += 1

print(f"Data cleaning finished. Removed {removed_count} invalid files.")

print(f"Deleted {num_skipped} corrupted or unsupported files.")

# โหลดข้อมูลและปรับขนาดภาพ
batch_size = 32
img_height = 224
img_width = 224

train_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

val_ds = tf.keras.utils.image_dataset_from_directory(
  data_dir,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

# ปรับสเกลสี Normalization ให้อยู่ในช่วง 0-1
normalization_layer = tf.keras.layers.Rescaling(1./255)
normalized_train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
normalized_val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

print("Data preparation complete.")

Data cleaning finished. Removed 6 invalid files.
Deleted 0 corrupted or unsupported files.
Found 9430 files belonging to 2 classes.
Using 7544 files for training.
Found 9430 files belonging to 2 classes.
Using 1886 files for validation.
Data preparation complete.


In [44]:
# โหลด Pre-trained Model (MobileNetV2)
# include_top=False คือการตัดเลเยอร์ทายทาย (หัว) ของโมเดลเดิมทิ้ง
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze เลเยอร์ของ base_model ไว้ ไม่ให้ค่า weights เดิมที่เทรนมาดีแล้วพังระหว่างที่เราเทรน
base_model.trainable = False

# สร้างโครงสร้างโมเดลของเรา (Build Model)
model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    # เลเยอร์สุดท้ายสำคัญมาก: ต้องมี 9 โหนด เท่ากับจำนวนคลาสของปลา และใช้ Softmax เพื่อคิดเป็น % ความน่าจะเป็น
    tf.keras.layers.Dense(9, activation='softmax')
])

# คอมไพล์โมเดล (Compile Model)
# กำหนด Optimizer (ตัวปรับน้ำหนัก), Loss Function (ตัววัดความผิดพลาด) และ Metrics (ตัววัดผล)
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

# แสดงโครงสร้างโมเดลทั้งหมด
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 9)              │         1,161 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,423,113 (9.24 MB)

 Trainable params: 165,129 (645.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [45]:
# กำหนดจำนวนรอบในการเทรน
epochs = 5

# เริ่มเทรนโมเดลโดยเก็บประวัติการเทรนไว้ในตัวแปร history
history = model.fit(
  normalized_train_ds,
  validation_data=normalized_val_ds,
  epochs=epochs
)

Epoch 1/5
236/236 ━━━━━━━━━━━━━━━━━━━━ 105s 363ms/step - accuracy: 0.9818 - loss: 0.0690 - val_accuracy: 0.9915 - val_loss: 0.0226
Epoch 2/5
236/236 ━━━━━━━━━━━━━━━━━━━━ 65s 274ms/step - accuracy: 0.9967 - loss: 0.0111 - val_accuracy: 0.9894 - val_loss: 0.0285
Epoch 3/5
236/236 ━━━━━━━━━━━━━━━━━━━━ 53s 224ms/step - accuracy: 0.9975 - loss: 0.0084 - val_accuracy: 0.9968 - val_loss: 0.0126
Epoch 4/5
236/236 ━━━━━━━━━━━━━━━━━━━━ 53s 225ms/step - accuracy: 0.9977 - loss: 0.0063 - val_accuracy: 0.9952 - val_loss: 0.0138
Epoch 5/5
236/236 ━━━━━━━━━━━━━━━━━━━━ 53s 225ms/step - accuracy: 0.9991 - loss: 0.0026 - val_accuracy: 0.9952 - val_loss: 0.0178


In [46]:
model.save('fish_model.keras')